# All Phases Combined Notebook

This notebook contains Phase 1, Phase 2 Supervised Learning, and Phase 2 Unsupervised Learning in one file.

The repository still includes separate notebooks in the required folders.


# SWE485 Project - Phase 1

# Medical Insurance Cost Risk and Advice System

## Group Information

| Student Name | Student ID | Responsibility |
|---|---:|---|
| Munthir Almukhif | 444101355 | Dataset loading and checking |
| Haitham Alduais | 444105932 | Summary and visualization |
| Mohammad Alfarraj | 443102025 | Preprocessing |


## Project Motivation

Medical insurance charges are different from one person to another. Age, BMI, smoking status, number of children, sex, and region may affect the final cost.

Our project uses this data to estimate medical insurance cost risk and give a simple cost-risk advice label: low, medium, or high.

## Problem Scope

The main target is `charges`, so the original task is regression. To make the project work as an advice system, we also create `risk_level` from the charges column.

The advice is only general cost-risk guidance. It is not medical advice.


## Dataset Goal and Source

Dataset name: Medical Cost Personal Dataset

Source: https://www.kaggle.com/datasets/mirichoi0218/insurance?resource=download

Goal: predict insurance charges using personal and health-related features.

## Columns

| Column | Type | Meaning |
|---|---|---|
| age | numeric | Age of the beneficiary |
| sex | categorical | male or female |
| bmi | numeric | Body Mass Index |
| children | numeric | Number of children/dependents |
| smoker | categorical | yes or no |
| region | categorical | US region |
| charges | numeric target | Medical insurance charges |


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv("Dataset/insurance.csv")
display(df.head())

In [ ]:

# Basic dataset information
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset info:")
df.info()


## General Information

The dataset has 7 columns. There are 6 input features and 1 original target variable.

Original target: `charges`

`charges` is continuous, so this is mainly a regression problem.

For the advice part, we create a new column called `risk_level`:

- Low Cost Risk
- Medium Cost Risk
- High Cost Risk


In [ ]:

# Statistical summary
numeric_cols = ["age", "bmi", "children", "charges"]
categorical_cols = ["sex", "smoker", "region"]

summary = df[numeric_cols].agg(["count", "mean", "var", "std", "min", "median", "max"]).T
print("Numeric statistical summary:")
display(summary.round(3))

print("Categorical value counts:")
for col in categorical_cols:
    print("\n" + col)
    print(df[col].value_counts())


In [ ]:

# Missing values and duplicate rows
missing_values = df.isna().sum()
display(missing_values.to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())


## Missing Values and Duplicates

We checked all columns for missing values. If the output shows zeros, then no imputation is needed.

We also checked duplicate rows. Duplicate rows are removed during preprocessing.


In [ ]:

# Create the advice target
# The original target charges is numeric.
# For advice, we split charges into three cost-risk groups.

risk_names = ["Low Cost Risk", "Medium Cost Risk", "High Cost Risk"]

df["risk_level"] = pd.qcut(
    df["charges"],
    q=3,
    labels=risk_names
)

print("Risk level distribution:")
display(df["risk_level"].value_counts().reindex(risk_names).to_frame("count"))


In [ ]:
# Data visualization
plt.figure(figsize=(7, 4))
plt.hist(df["charges"], bins=30)
plt.title("Distribution of Insurance Charges")
plt.xlabel("Charges")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(6, 4))
df["smoker"].value_counts().plot(kind="bar")
plt.title("Smoker Count")
plt.xlabel("Smoker")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(x="smoker", y="charges", data=df)
plt.title("Charges by Smoking Status")
plt.xlabel("Smoker")
plt.ylabel("Charges")
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(x="age", y="charges", hue="smoker", data=df)
plt.title("Age vs Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(x="bmi", y="charges", hue="smoker", data=df)
plt.title("BMI vs Charges")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.show()

plt.figure(figsize=(6, 4))
df["risk_level"].value_counts().reindex(risk_names).plot(kind="bar")
plt.title("Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Count")
plt.xticks(rotation=15)
plt.show()

## Visualization Notes

From the plots, we can study the main relationships in the data.

The `charges` column is not evenly distributed. Some people have very high charges.

Smoking status is important because smokers usually have higher charges.

Age and BMI are also useful features because they may be related to insurance cost.

The risk level plot checks if the advice classes are balanced.


## Preprocessing Plan

The preprocessing steps are:

1. Remove duplicate rows if any exist.
2. Encode `sex` as 0 and 1.
3. Encode `smoker` as 0 and 1.
4. One-hot encode `region` because it has no natural order.
5. Standardize `age`, `bmi`, and `children` for scale-sensitive models.

We do not scale `charges` in Phase 1 because it is the prediction target.


In [ ]:

# Preprocessing

df_clean = df.drop_duplicates().copy()

# Make sure risk_level exists after removing duplicates
df_clean["risk_level"] = pd.qcut(
    df_clean["charges"],
    q=3,
    labels=risk_names
)

# Binary encoding
df_clean["sex"] = df_clean["sex"].map({"female": 0, "male": 1})
df_clean["smoker"] = df_clean["smoker"].map({"no": 0, "yes": 1})

# One-hot encoding for region
df_model = pd.get_dummies(df_clean, columns=["region"], drop_first=True, dtype=int)

# Standardize selected numerical features only
scale_cols = ["age", "bmi", "children"]
df_scaled = df_model.copy()

for col in scale_cols:
    df_scaled[col] = (df_scaled[col] - df_scaled[col].mean()) / df_scaled[col].std()

print("Preprocessed data preview:")
display(df_model.head())

print("Scaled data preview:")
display(df_scaled.head())


In [ ]:
# Prepare features and targets for Phase 2
X = df_scaled.drop(columns=["charges", "risk_level"])
y_regression = df_scaled["charges"]
y_classification = df_scaled["risk_level"]

print("Feature matrix shape:", X.shape)
print("Regression target shape:", y_regression.shape)
print("Classification target shape:", y_classification.shape)

# Save the processed data
df_scaled.to_csv("Dataset/insurance_preprocessed_phase1.csv", index=False)
print("Saved: Dataset/insurance_preprocessed_phase1.csv")

## Phase 1 Findings

The dataset was loaded successfully and contains numerical and categorical features.

There are no missing values in the dataset.

The target `charges` is continuous, so the main task is regression.

We created `risk_level` to make the system give simple advice labels.

Categorical values were converted to numbers so machine learning models can use them.

The cleaned data is now ready for Phase 2.


# Phase 2 - Supervised Learning

## Project Title

Medical Insurance Cost Risk and Advice System

## Goal

The goal of this notebook is to build supervised machine learning models that predict the user's insurance cost-risk advice class.

The original column `charges` is continuous, so we convert it into three advice classes:

- Low Cost Risk
- Medium Cost Risk
- High Cost Risk

This makes the model more suitable for an advice system.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")


In [ ]:
# Load dataset
df = pd.read_csv("Dataset/insurance.csv")
display(df.head())

In [ ]:

# Basic cleaning
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


## Target for Supervised Learning

The original target is `charges`, which is a number. For the advice system, we create `risk_level` by dividing charges into three groups.

This gives the model a clear advice class to predict.


In [ ]:

# Create advice classes from charges
risk_names = ["Low Cost Risk", "Medium Cost Risk", "High Cost Risk"]

df["risk_level"] = pd.qcut(
    df["charges"],
    q=3,
    labels=risk_names
)

print("Risk level distribution:")
print(df["risk_level"].value_counts().reindex(risk_names))

plt.figure(figsize=(6, 4))
df["risk_level"].value_counts().reindex(risk_names).plot(kind="bar")
plt.title("Risk Level Class Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.show()


## Algorithm Selection

We selected three supervised models:

1. Logistic Regression: a simple baseline model.
2. Random Forest: useful for non-linear relations and mixed features.
3. SVM: useful when the decision boundary is not linear.

We compare them using the same train-test split and the same evaluation metrics.


In [ ]:

# Prepare features and target
features = ["age", "sex", "bmi", "children", "smoker", "region"]
X = df[features]

y = df["risk_level"].astype(str).map({name: i for i, name in enumerate(risk_names)})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


In [ ]:

# Train and compare models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)
}

results = []
trained_models = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision Macro": precision_score(y_test, y_pred, average="macro"),
        "Recall Macro": recall_score(y_test, y_pred, average="macro"),
        "F1 Macro": f1_score(y_test, y_pred, average="macro"),
        "ROC-AUC Macro": roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
        "CV F1 Mean": cv_f1.mean(),
        "CV F1 Std": cv_f1.std()
    })

    trained_models[model_name] = pipe

results_df = pd.DataFrame(results).sort_values(by="F1 Macro", ascending=False)
display(results_df.round(4))

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best model based on F1 Macro:", best_model_name)


In [ ]:

# Detailed report for the best model
y_best_pred = best_model.predict(X_test)

print("Classification report for:", best_model_name)
print(classification_report(y_test, y_best_pred, target_names=risk_names))

cm = confusion_matrix(y_test, y_best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=risk_names)
disp.plot(cmap="Blues", xticks_rotation=30)
plt.title("Confusion Matrix - " + best_model_name)
plt.show()


In [ ]:

# Compare model scores visually
plt.figure(figsize=(7, 4))
plt.bar(results_df["Model"], results_df["F1 Macro"])
plt.title("Model Comparison by F1 Macro")
plt.xlabel("Model")
plt.ylabel("F1 Macro")
plt.ylim(0, 1)
plt.show()


In [ ]:

# Feature importance for Random Forest
# This helps us understand which features affected the prediction.

rf_pipe = trained_models["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Top feature importances from Random Forest:")
display(importance_df.head(10))

plt.figure(figsize=(8, 4))
plt.barh(importance_df.head(10)["Feature"], importance_df.head(10)["Importance"])
plt.title("Top 10 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.show()


## Prediction Example

This part shows how the final system can receive user input and return an advice class.

The advice is general and based on the predicted insurance cost-risk level.


In [ ]:

# Example user input
sample_user = pd.DataFrame({
    "age": [32],
    "sex": ["male"],
    "bmi": [29.5],
    "children": [1],
    "smoker": ["no"],
    "region": ["northwest"]
})

pred_num = best_model.predict(sample_user)[0]
pred_label = risk_names[pred_num]
probs = best_model.predict_proba(sample_user)[0]

print("Sample user:")
display(sample_user)

print("Predicted advice class:", pred_label)
print("Prediction probabilities:")
for name, prob in zip(risk_names, probs):
    print(name + ":", round(prob, 3))


def give_advice(label):
    if label == "Low Cost Risk":
        return "The user is predicted to have low insurance cost risk compared with the dataset."
    elif label == "Medium Cost Risk":
        return "The user is predicted to have a medium insurance cost risk."
    else:
        return "The user is predicted to have high insurance cost risk. Major factors should be reviewed."

print("\nAdvice:")
print(give_advice(pred_label))


## Results Interpretation

The best model is selected using F1 Macro because the target has three classes and we want balanced performance across all classes.

From the comparison table, the best model is the one with the highest F1 Macro score.

Random Forest is also useful because it can capture non-linear relationships between age, BMI, smoking status, and insurance cost risk.

The feature importance chart helps explain the model. Important features are expected to include smoking status, age, and BMI.

The prediction example shows how the model can be used as an advice system. The user enters basic information, and the model returns a cost-risk class with a simple advice message.


# Phase 2 - Unsupervised Learning

## Goal

The goal of this notebook is to use clustering to group users into similar profiles.

We use K-Means clustering because it is simple and useful for grouping users based on numerical and encoded categorical features. The target label is removed before clustering.

After clustering, we check the clusters using:

- WCSS
- Silhouette Score
- BCubed Precision and Recall
- PCA visualization
- Cluster profile summary


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")


In [ ]:
# Load dataset
df = pd.read_csv("Dataset/insurance.csv")
df = df.drop_duplicates().reset_index(drop=True)
display(df.head())

In [ ]:

# Create risk_level only for later checking.
# It will not be used while training the clustering model.

risk_names = ["Low Cost Risk", "Medium Cost Risk", "High Cost Risk"]

df["risk_level"] = pd.qcut(
    df["charges"],
    q=3,
    labels=risk_names
)

print(df["risk_level"].value_counts().reindex(risk_names))


## Preparing Data for Clustering

For clustering, we remove the class label and target column. We only use user information:

- age
- sex
- bmi
- children
- smoker
- region

This follows the requirement that the class label should be removed before clustering.


In [ ]:

# Prepare features for clustering
cluster_features = ["age", "sex", "bmi", "children", "smoker", "region"]
X_cluster_raw = df[cluster_features]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocess_cluster = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_cluster = preprocess_cluster.fit_transform(X_cluster_raw)

# Convert to dense array for PCA and clustering display
if hasattr(X_cluster, "toarray"):
    X_cluster = X_cluster.toarray()

print("Clustering feature matrix shape:", X_cluster.shape)


## Choosing Number of Clusters

We test different values of K. WCSS helps us see the elbow point. Silhouette Score helps us measure how well separated the clusters are.


In [ ]:

# Test different K values
k_values = range(2, 9)
wcss = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X_cluster)

    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster, cluster_labels))

score_df = pd.DataFrame({
    "K": list(k_values),
    "WCSS": wcss,
    "Silhouette Score": silhouette_scores
})

display(score_df.round(4))

best_k = score_df.sort_values(by="Silhouette Score", ascending=False).iloc[0]["K"]
best_k = int(best_k)
print("Best K based on Silhouette Score:", best_k)


In [ ]:

# Plot WCSS and Silhouette Score
plt.figure(figsize=(6, 4))
plt.plot(score_df["K"], score_df["WCSS"], marker="o")
plt.title("Elbow Method - WCSS")
plt.xlabel("Number of Clusters K")
plt.ylabel("WCSS")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(score_df["K"], score_df["Silhouette Score"], marker="o")
plt.title("Silhouette Score by K")
plt.xlabel("Number of Clusters K")
plt.ylabel("Silhouette Score")
plt.show()


In [ ]:

# Train final K-Means model

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
clusters = kmeans_final.fit_predict(X_cluster)

df_clustered = df.copy()
df_clustered["cluster"] = clusters

print("Final WCSS:", round(kmeans_final.inertia_, 2))
print("Final Silhouette Score:", round(silhouette_score(X_cluster, clusters), 4))
print("Cluster counts:")
print(df_clustered["cluster"].value_counts().sort_index())


In [ ]:

# BCubed Precision and Recall
# We use risk_level only for evaluation after clustering.
# risk_level was not used when training K-Means.

def bcubed_precision_recall(true_labels, cluster_labels):
    true_labels = np.array(true_labels)
    cluster_labels = np.array(cluster_labels)

    precision_scores = []
    recall_scores = []

    for i in range(len(true_labels)):
        same_cluster = cluster_labels == cluster_labels[i]
        same_class = true_labels == true_labels[i]

        correct = np.sum(same_cluster & same_class)

        precision_i = correct / np.sum(same_cluster)
        recall_i = correct / np.sum(same_class)

        precision_scores.append(precision_i)
        recall_scores.append(recall_i)

    return np.mean(precision_scores), np.mean(recall_scores)


bcubed_precision, bcubed_recall = bcubed_precision_recall(
    df_clustered["risk_level"],
    df_clustered["cluster"]
)

print("BCubed Precision:", round(bcubed_precision, 4))
print("BCubed Recall:", round(bcubed_recall, 4))


In [ ]:

# Cluster profile summary
# This helps us understand what each cluster means.

cluster_summary = df_clustered.groupby("cluster").agg(
    count=("cluster", "size"),
    avg_age=("age", "mean"),
    avg_bmi=("bmi", "mean"),
    avg_children=("children", "mean"),
    smoker_rate=("smoker", lambda x: (x == "yes").mean()),
    avg_charges=("charges", "mean")
)

print("Cluster profile summary:")
display(cluster_summary.round(3))

risk_by_cluster = pd.crosstab(
    df_clustered["cluster"],
    df_clustered["risk_level"],
    normalize="index"
) * 100

print("Risk level percentage inside each cluster:")
display(risk_by_cluster.round(2))


In [ ]:

# PCA visualization of clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, s=30)
plt.title("K-Means Clusters using PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

print("Explained variance by PCA components:", np.round(pca.explained_variance_ratio_, 3))


## Unsupervised Results Interpretation

K-Means clustering was used to group users into similar profiles based on age, sex, BMI, children, smoking status, and region.

The class label `risk_level` was removed before training the clustering model. This is important because clustering should find groups without using the target label.

We tested different values of K using WCSS and Silhouette Score. The best K is selected based on the highest Silhouette Score.

After clustering, we compared the clusters with `risk_level` only for evaluation and interpretation.

BCubed Precision and Recall were used to check how well the clusters matched the risk-level groups.

The cluster profile summary helps explain the meaning of each cluster. For example, a cluster with a higher smoker rate or higher average BMI may represent users with higher insurance cost risk.

These clusters can improve the advice system by creating user profiles. The system can use both the supervised model prediction and the cluster profile to give better cost-risk advice.
